#                                            DIRECTIONAL DRILLING 
##                                           *DIRECTIONAL WELLS PROFILES AND DIRECTIONAL WELLS TRAJECTORIES*

***

![well](Resources/Well_prof.jpg)

# Python Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import namedtuple
from math import radians, isclose, acos, asin, cos, sin, tan, atan, degrees, sqrt

# *Directional Wells Profiles*

## *Slant Well Profile (J Type)*

![j](Resources/j_prof.png)

In [2]:
Data = namedtuple("Input", "TVD KOP BUR DH")
Output = namedtuple("Output", "R Theta TVD_EOB Md_EOB Dh_EOB Tan_len Md_total")

def well_J(data:Data, unit='ingles') -> Output:
    tvd = data.TVD
    kop = data.KOP
    bur = data.BUR
    dh = data.DH
    if unit == 'ingles':
        R = 5729.58 / bur
    else:
        R = 1718.87 / bur

    # segment DC
    if dh > R:
        dc = dh - R
    elif dh < R:
        dc =   R - dh

    # Segment DO
    do = tvd - kop

    # DOC Angle
    doc = degrees(atan(dc / do))

    # Segment OC
    oc = sqrt(dc**2 + do**2)

    #BOC Angle
    boc = degrees(acos(R / oc))

    # BOD Angle
    if R < dh:
        bod = boc - doc
    elif R > dh:
        bod = boc + doc

    # Theta angle
    theta = 90 - bod

    # TVD a EOB
    tvd_eob = kop + abs(R * sin(radians(theta)))

    # MD a EOB
    if unit == 'ingles':
        md_eob = kop + (theta / bur) * 100
    else:
        md_eob = kop + (theta / bur) * 30

    # DH a EOB
    dh_eob = R - R * cos(radians(theta))

    # Tangent section
    tan_len = sqrt(oc**2 - R**2)

    # MD Total
    if unit == 'ingles':
        md_total = kop + (theta / bur) * 100 + tan_len
    else:
        md_total = kop + (theta / bur) * 30 + tan_len

        
    return Output(R=R, Theta=theta, TVD_EOB=tvd_eob, Md_EOB=md_eob, Dh_EOB=dh_eob, \
                  Tan_len=tan_len, Md_total=md_total)

## *Ejercicio 1*

In [3]:
# data
tvd = 8000 #ft
kop = 500 #ft
bur = 2 #o/100ft
dh = 970.8 #ft

In [4]:
trajectory_J = well_J(Data(tvd, kop, bur, dh))
trajectory_J

Output(R=2864.79, Theta=7.564230623470863, TVD_EOB=877.1139517978513, Md_EOB=878.2115311735431, Dh_EOB=24.929649303260703, Tan_len=7185.414140882904, Md_total=8063.625672056447)

In [6]:
names = ['R', 'theta', 'tvd_EOB', 'Md_EOB', 'Dh_EOB', 'Lengh_tan', 'Md_Total']
for param, value in zip(names, trajectory_J):
    if param == 'theta':
        print(f"{param} : {value:.3f} degrees")
    else:
        print(f"{param} : {value:.3f} ft")

R : 2864.790 ft
theta : 7.564 degrees
tvd_EOB : 877.114 ft
Md_EOB : 878.212 ft
Dh_EOB : 24.930 ft
Lengh_tan : 7185.414 ft
Md_Total : 8063.626 ft


## *S-Type Well Profile*

![s](Resources/s_prof.png)

In [ ]:
# Function for S-Type wells

# Function to calculate parameters from a S-Type well
Data_S = namedtuple("Input", "TVD KOP BUR DOR DH")
Output_S = namedtuple("Output", "R1 R2 Theta TVD_EOB Md_EOB Dh_EOB Tan_len Md_SOD TVD_SOD Dh_SOD Md_total")


# *Ejercicio 2*

In [ ]:
# Data
kop = 6084 #ft
tvd = 12000 #ft
bur = 3 #o/100ft
dor = 2 #o/ft
dh = 3500 #ft

## *Horizontal Well Profiles*

![hor](Resources/Horizontal_prof.jpg)

In [3]:
# Function for horizontal wells

# Function to calculate parameters of a Horizontal Well
Data_H = namedtuple("Input", "TVD KOP BUR1 BUR2 DH")
Output_H = namedtuple("Output", "R1 R2 Theta TVD_EOB1 Md_EOB1 Dh_EOB1 Tan_len Md_SOB2 Md_total")

def Well_H(data_h:Data_H, unit="ingles")-> Output_H:
    tvd = data_h.TVD
    kop = data_h.KOP
    bur1 = data_h.BUR1
    bur2 = data_h.BUR2
    dh = data_h.DH
    if unit == 'ingles':
        R1 = 5729.58 / bur1
        R2 = 5729.58 / bur2
    else:
        R1 = 1718.87 / bur1
        R2 = 1718.87 / bur2

    EG = (tvd - kop) - R2
    EO = dh - R1
    A_GOE = np.arctan(EG/EO)*180/np.pi
    OG = (EG**2 + EO**2)**0.5
    OF = R1 - R2
    A_GOF = np.arccos(OF/OG)*180/np.pi
    A_AOB = 180 - A_GOE - A_GOF                                  #Maximo angulo de construccion
    TVD_V2 = kop + R1*np.sin(A_AOB*np.pi/180)                        #TVD hasta end of build 1
    if unit == 'ingles':
        MD_EOB1 = kop + (A_AOB/bur1)*100  # MD hasta end of build
    else:
        MD_EOB1 = kop + (A_AOB/bur1)*30

    D1 = R1 - R1*np.cos(A_AOB*np.pi/180)                             #Desplazamiento horizontal hasta end of build 1
    BC = (OG**2 - OF**2)**0.5
    MD_SOB2 = MD_EOB1 + BC                                            # MD hasta start of build 2
    TVD_V3 = TVD_V2 + BC*np.cos(A_AOB*np.pi/180)                     # TVD hasta start of drop
    D2 = D1 + BC*np.sin(A_AOB*np.pi/180)                              # Distancia horizontal hasta start of build 2
    A_GCD = 90 - (90 - A_GOF) - (90 - A_GOE)
    if unit == 'ingles':
         MDT = MD_SOB2 + (A_GCD/bur2)*100    # MD hasta end of build
    else:
         MDT = MD_SOB2 + (A_GCD/bur2)*30

    return Output(R1=R1, R2= R2,  Theta=A_AOB, TVD_EOB1=TVD_V2, Md_EOB1=MD_EOB1, Dh_EOB1=D1, Tan_len=BC, Md_SOB2=MD_SOB2, Md_total=MDT)

## *Ejercicio 3*

In [4]:
# Data
tvd = 3800 #ft
kop = 2000 #ft
bur1 = 5.73 #o/100ft
bur2 = 9.55 #o/100ft
dh = 1800 #ft

In [6]:
trajetory_H= Well_H(Data_H(tvd, kop, bur1, bur2, dh))
trajetory_H


name = ['R1', "R2", 'Theta', 'TVD_EOB1', 'Md_EOB1', 'Dh_EOB1', 'Tan_len', 'Md_SOB2',"Md_total"]
for param, value in zip(names, trajectory_J):
    if param == 'Theta':
        print(f"{param} : {value:.3f} degrees")
    else:
        print(f"{param} : {value:.3f} ft")

NameError: name 'Output' is not defined

***